# Chapter 4: Implementing a GPT model from Scratch To Generate Text 

In [ ]:
from importlib.metadata import version

print("matplotlib version:", version("matplotlib"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

- In this chapter, I implement a GPT-like LLM architecture; the next chapter will focus on training this LLM

*[Figure from the book — see book or companion site for diagram.]*

&nbsp;
## 4.1 Coding an LLM architecture

- My framing here: GPT/Llama-style models generate text token by token using the decoder pathway from the original Transformer.
- I treat these as decoder-style LLMs in my notes.
- Compared with many classic deep-learning models, the big difference is parameter scale rather than code complexity.
- I should keep watching for repeated block patterns because transformer stacks reuse the same core modules.

*[Figure from the book — see book or companion site for diagram.]*

- Earlier chapters used tiny dimensions to keep examples readable; here I shift to GPT-2-like settings to build better intuition for practical shapes.
- In this notebook I implement the smallest GPT-2 variant as my baseline architecture.
- I’m following the original GPT-2 references while keeping in mind the common 117M vs 124M parameter naming confusion.
- Later chapters will let me load pretrained checkpoints across larger GPT-2 sizes with the same implementation skeleton.

- Configuration details for the 124 million parameter GPT-2 model include:

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

- I keep config key names short so later code is easier to scan.
- `"vocab_size"`: tokenizer vocabulary size for the model’s output space.
- `"context_length"`: maximum number of tokens the model can process at once.
- `"emb_dim"`: width of each token representation vector.
- `"n_heads"`: number of attention heads used per transformer block.
- `"n_layers"`: total transformer block count in the stack.
- `"drop_rate"`: dropout strength used during training for regularization.
- `"qkv_bias"`: whether Q/K/V linear projections include bias terms.

*[Figure from the book — see book or companion site for diagram.]*

In [ ]:
import torch
import torch.nn as nn


class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The parameters here are just to mimic the LayerNorm interface.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x

*[Figure from the book — see book or companion site for diagram.]*

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [ ]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

---

**Note**

- If your printed tensor differs across platforms, that is expected for this demo run.

```
Output shape: torch.Size([2, 4, 50257])
tensor([[[-0.9289,  0.2748, -0.7557,  ..., -1.6070,  0.2702, -0.5888],
         [-0.4476,  0.1726,  0.5354,  ..., -0.3932,  1.5285,  0.8557],
         [ 0.5680,  1.6053, -0.2155,  ...,  1.1624,  0.1380,  0.7425],
         [ 0.0447,  2.4787, -0.8843,  ...,  1.3219, -0.0864, -0.5856]],

        [[-1.5474, -0.0542, -1.0571,  ..., -1.8061, -0.4494, -0.6747],
         [-0.8422,  0.8243, -0.1098,  ..., -0.1434,  0.2079,  1.2046],
         [ 0.1355,  1.1858, -0.1453,  ...,  0.0869, -0.1590,  0.1552],
         [ 0.1666, -0.8138,  0.2307,  ...,  2.5035, -0.3055, -0.3083]]],
       grad_fn=<UnsafeViewBackward0>)
```

- These are random initialization outputs, so small differences are fine and do not block the chapter flow.
- One likely source is backend-dependent dropout behavior across OS/build combinations, as discussed in the linked PyTorch issue.

---

&nbsp;
## 4.2 Normalizing activations with layer normalization

- Layer normalization re-centers activations and rescales them, which usually makes optimization more stable.
- In my mental model, it helps gradients behave more predictably across depth.
- In this GPT stack, normalization appears around attention/feed-forward sublayers and again near the output path.

*[Figure from the book — see book or companion site for diagram.]*

- I'll see how layer normalization works by passing a small input sample through a simple neural network layer:

In [ ]:
torch.manual_seed(123)

# create 2 training examples with 5 dimensions (features) each
batch_example = torch.randn(2, 5) 

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

- I'll compute the mean and variance for each of the 2 inputs above:

In [ ]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

- I apply normalization per sample (row), and `dim=-1` means the stats are computed across the feature axis for each row independently.

*[Figure from the book — see book or companion site for diagram.]*

- Subtracting the row mean and dividing by the row standard deviation re-centers each sample near 0 with variance near 1 across features:

In [ ]:
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs:\n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

- Each input is centered at 0 and has a unit variance of 1; to improve readability, I can disable PyTorch's scientific notation:

In [ ]:
torch.set_printoptions(sci_mode=False)
print("Mean:\n", mean)
print("Variance:\n", var)

- Above, we normalized the features of each input
- Now, using the same idea, I can implement a `LayerNorm` class:

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

**Scale and shift**

- In this implementation, normalization is followed by two learnable vectors: `scale` and `shift`.
- They start as identity behavior (`scale=1`, `shift=0`), but training can move them when that improves the objective.
- I think of this as giving the model a controlled way to recover useful feature ranges after normalization.
- I also add a small `eps` term before the square root to avoid numerical instability when variance is tiny.

**Biased variance**
- Setting `unbiased=False` uses variance with denominator $n$ instead of $n-1$.
- For large embedding widths, the practical difference is usually very small.
- I keep this choice here to stay aligned with GPT-2-style layer-norm behavior for later weight compatibility checks.

- Next I test this `LayerNorm` class on the sample activations.

In [ ]:
ln = LayerNorm(emb_dim=6)
out_ln = ln(out)

In [ ]:
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

- Variance is not exactly 1 because I use `eps`

*[Figure from the book — see book or companion site for diagram.]*

&nbsp;
## 4.3 Implementing a feed forward network with GELU activations

- This section builds the feed-forward submodule used inside each transformer block.
- I start by reviewing the activation choice because it strongly affects representation quality.
- ReLU is simple and widely used, but GPT-style models usually prefer smoother alternatives.
- Two common alternatives I want to contrast are GELU and SwiGLU.
- My takeaway: these smoother/gated activations often improve optimization behavior in deep transformer stacks.

- I can define GELU as $\mathrm{GELU}(x)=x\,\Phi(x)$, where $\Phi$ is the standard normal CDF.
- In practice I usually use the faster tanh approximation instead:
$$
\mathrm{GELU}(x) \approx 0.5x\left(1+\tanh\left(\sqrt{\frac{2}{\pi}}\,(x+0.044715x^3)\right)\right)
$$
- This is the approximation used in GPT-2 and it is sufficient for my implementation practice.

In [ ]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [ ]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

# Some sample data
x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

- ReLU stays piecewise linear: positive values pass through, negative values clamp to zero.
- GELU changes more smoothly, including a softer transition on negative inputs, which is one reason it is favored in transformer models.

- Next I implement the `FeedForward` module that will sit inside each transformer block.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
print(GPT_CONFIG_124M["emb_dim"])

*[Figure from the book — see book or companion site for diagram.]*

In [ ]:
ffn = FeedForward(GPT_CONFIG_124M)

# input shape: [batch_size, num_token, emb_size]
x = torch.rand(2, 3, 768) 
out = ffn(x)
print(out.shape)

*[Figure from the book — see book or companion site for diagram.]*

*[Figure from the book — see book or companion site for diagram.]*

&nbsp;
## 4.4 Adding shortcut connections

- Here I review residual (shortcut) connections before wiring the full transformer block.
- The core idea is to give gradients a shorter route through deep stacks, which helps optimization stay stable.
- Practically, I add earlier activations back to later outputs when tensor shapes match.
- I’ll demo this with a tiny network before reusing the same pattern in GPT blocks.

*[Figure from the book — see book or companion site for diagram.]*

- In code, it looks like this:

In [ ]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x


def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([[0.]])

    # Calculate loss based on how close the target
    # and output are
    loss = nn.MSELoss()
    loss = loss(output, target)
    
    # Backward pass to calculate the gradients
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            # Print the mean absolute gradient of the weights
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

- I'll print the gradient values first **without** shortcut connections:

In [ ]:
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

- Next, I'll print the gradient values **with** shortcut connections:

In [ ]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

- As I can see based on the output above, shortcut connections prevent the gradients from vanishing in the early layers (towards `layer.0`)
- I will use this concept of a shortcut connection next when we implement a transformer block

&nbsp;
## 4.5 Connecting attention and linear layers in a transformer block

- This section combines the building blocks into a full transformer block.
- I merge causal self-attention, feed-forward layers, normalization, dropout, and residual paths.
- My goal is to keep dimensions consistent while enriching token representations across the stack.

In [ ]:
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch03 import MultiHeadAttention

from previous_chapters import MultiHeadAttention


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x

*[Figure from the book — see book or companion site for diagram.]*

- For a batch of token embeddings, the block preserves overall shape while transforming each token with attention + MLP updates.
- I interpret the output as a context-enriched version of the incoming token states.

In [ ]:
torch.manual_seed(123)

x = torch.rand(2, 4, 768)  # Shape: [batch_size, num_tokens, emb_dim]
block = TransformerBlock(GPT_CONFIG_124M)
output = block(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

*[Figure from the book — see book or companion site for diagram.]*

&nbsp;
## 4.6 Coding the GPT model

- Now I plug this transformer block into the earlier GPT skeleton so the model is actually usable end-to-end.
- For the 124M-style setup, the same block is stacked 12 times.

*[Figure from the book — see book or companion site for diagram.]*

- The corresponding code implementation, where `cfg["n_layers"] = 12`:

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

- Using the configuration of the 124M parameter model, I can now instantiate this GPT model with random initial weights as follows:

In [ ]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

- I will train this model in the next chapter
- However, a quick note about its size: we previously referred to it as a 124M parameter model; I can double check this number as follows:

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

- The raw parameter count here is higher than the 124M label, so I sanity-check why.
- Classic GPT-2 uses weight tying, meaning output projection weights are shared with token embeddings.
- Without tying, embedding and output projection both contribute full parameter sets.
- I inspect both matrix shapes next to confirm that symmetry.
- The quick recount below helps me compare tied vs untied configurations clearly.

In [ ]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
print("Output layer shape:", model.out_head.weight.shape)

- If I assume GPT-2-style weight tying, the effective trainable parameter total drops.
- Subtracting the standalone output-head parameters gives the tied-style count estimate.

In [ ]:
total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

- For this learning implementation I keep untied weights because it is easier to inspect while debugging.
- I will revisit tied weights later when loading pretrained checkpoints for compatibility experiments.
- I also compute memory footprint here as a practical sizing reference.

In [ ]:
# Calculate the total size in bytes (assuming float32, 4 bytes per parameter)
total_size_bytes = total_params * 4

# Convert to megabytes
total_size_mb = total_size_bytes / (1024 * 1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")

- Practice idea: try the other GPT-2 family configurations and compare parameter counts + memory.

- **GPT2-small** (baseline in this notebook):
  - "emb_dim" = 768
  - "n_layers" = 12
  - "n_heads" = 12

- **GPT2-medium:**
  - "emb_dim" = 1024
  - "n_layers" = 24
  - "n_heads" = 16

- **GPT2-large:**
  - "emb_dim" = 1280
  - "n_layers" = 36
  - "n_heads" = 20

- **GPT2-XL:**
  - "emb_dim" = 1600
  - "n_layers" = 48
  - "n_heads" = 25

&nbsp;
## 4.7 Generating text

- GPT-style language generation is autoregressive: the model predicts the next token one step at a time.

*[Figure from the book — see book or companion site for diagram.]*

- `generate_text_simple` below uses greedy decoding for a clear baseline.
- At each step, it picks the token with the highest score as the next token.
- In the next chapter I’ll move to richer sampling methods beyond greedy decoding.
- The figure illustrates this iterative next-token loop from an initial context.

*[Figure from the book — see book or companion site for diagram.]*

In [ ]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):
        
        # Crop current context if it exceeds the supported context size
        # E.g., if LLM supports only 5 tokens, and the context size is 10
        # then only the last 5 tokens are used as context
        idx_cond = idx[:, -context_size:]
        
        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)
        
        # Focus only on the last time step
        # (batch, n_tokens, vocab_size) becomes (batch, vocab_size)
        logits = logits[:, -1, :]  

        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

- `generate_text_simple` appends one predicted token per iteration to grow the sequence.

*[Figure from the book — see book or companion site for diagram.]*

- I'll prepare an input example:

In [ ]:
start_context = "Hello, I am"

encoded = tokenizer.encode(start_context)
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)
print("encoded_tensor.shape:", encoded_tensor.shape)

In [ ]:
model.eval() # disable dropout

out = generate_text_simple(
    model=model,
    idx=encoded_tensor, 
    max_new_tokens=6, 
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Output:", out)
print("Output length:", len(out[0]))

- Remove batch dimension and convert back into text:

In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

- Note that the model is untrained; hence the random output texts above
- I will train the model in the next chapter

&nbsp;
## Summary and takeaways

- I keep a compact standalone implementation in [./gpt.py](./gpt.py) that mirrors the model built step-by-step in this notebook.
- I keep my chapter exercises in [./exercise-solutions.ipynb](./exercise-solutions.ipynb) for quick revision and checks.